In [1]:
from dotenv import load_dotenv
load_dotenv()

from langchain.chat_models import init_chat_model
from langchain.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage


In [52]:

MODEL = "nvidia/nemotron-3.5-super-vl:free"

In [3]:

@tool
def get_product_price(product_name: str) -> float:
    """
    Get the price of a product by its name.
    Supported products: laptop, mobile, headphone.
    """

    products = {
        "laptop": 1200.99,
        "mobile": 800.23,
        "headphone": 150.22
    }

    product_name = product_name.lower().strip()

    if product_name in products:
        return (products[product_name])

    return "Product not found"

In [25]:
get_product_price.invoke(
    {"product_name": "laptop"}
)

1200.99

In [28]:
get_product_price.invoke(
    {
       "product_name" : "car" 
    }
)

'Product not found'

In [29]:
print(get_product_price.name)
print(get_product_price.description)
print(get_product_price.args)

get_product_price
Get the price of a product by its name.
Supported products: laptop, mobile, headphone.
{'product_name': {'title': 'Product Name', 'type': 'string'}}


In [30]:
@tool
def apply_discount(discount_type: str, price: float) -> float:
    """
    Apply a discount to a product price.

    Supported discount types:
    - gold: 23%
    - silver: 12%
    - bronze: 5%

    Args:
        discount_type: The discount category.
        price: The original product price.

    Returns:
        The final price after applying the discount.
    """

    discounts = {
        "gold": 0.23,
        "silver": 0.12,
        "bronze": 0.05
    }

    discount_type = discount_type.lower().strip()

    if discount_type not in discounts:
        return "Invalid discount type"

    final_price = price * (1 - discounts[discount_type])

    return round(final_price, 2)

In [32]:

print(apply_discount.args)

{'discount_type': {'title': 'Discount Type', 'type': 'string'}, 'price': {'title': 'Price', 'type': 'number'}}


In [37]:
apply_discount.invoke(
    {
        "discount_type" : "GOLD",
        "price": 1000
    }
)

770.0

In [110]:
from langsmith import traceable
import os
@traceable(name="LangChain agent loop")
def run_agent(question: str):

    tools = [get_product_price , apply_discount]
    tools_by_name = {t.name:t for t in tools}

    llm = init_chat_model(
    model=MODEL,
    model_provider="openai",
    api_key=os.getenv("OpenRouter_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    temperature=0
)
    model_with_tools = llm.bind_tools(tools)

    print(f"Question is : {question} ")
    print("=" * 40)

    messages = [
        SystemMessage(
           content="""You are a shopping assistant. Your job is to help customers find product prices and calculate final prices after discounts.

You have access to two tools:
- get_product_price: Retrieves the original price of a product.
- apply_discount: Applies a discount to a price and returns the final price.

Rules:

1-Always use apply_discount to calculate the discounted price.
2- Always use get_product_price to retrieve the product's original price.
3. Never guess, assume, or invent tool arguments. Only use information explicitly provided by the customer or returned by previous tool calls.
4. If required information (product name or discount type) is missing, ask the customer before proceeding.
5. Never perform calculations yourself. All price calculations must be handled by the tools.
6. Never invent prices or tool results.
7. If the arguments is not valid ask the user to resend the right argument, give it the valid arguments
After succesfully using both tools, report the product name, original price, discount type, and final price to the customer.
"""
           ),
        HumanMessage(
            content=question
        )
            
    ]
    while True:

        # 1. Send messages to the model
        response = model_with_tools.invoke(messages)

        # 2. Add AI response to conversation history
        messages.append(response)

        # 3. Check whether the model requested any tools
        if not response.tool_calls:
            print("The final LLM response:")
            print(response.content)
            print()
            for i , message in enumerate(messages):
              print(f"The message {i+1} is :")
              print(message)
              print()
            break

        # 4. Execute requested tools
        for tool_call in response.tool_calls:

            selected_tool = tools_by_name[tool_call["name"]]

            result = selected_tool.invoke(tool_call["args"])

            # 5. Send tool result back to the conversation
            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=tool_call["id"]
                )
            )
    

In [ ]:
run_agent("What is the price of laptop after applying gold discount?")

Question is : What is the price of laptop after applying gold discount? 
The final LLM response:
Product: Laptop  
Original Price: $1200.99  
Discount Type: Gold  
Final Price: $924.76

The message1 is :
content="You are a shopping assistant. Your job is to help customers find product prices and calculate final prices after discounts.\n\nYou have access to two tools:\n- get_product_price: Retrieves the original price of a product.\n- apply_discount: Applies a discount to a price and returns the final price.\n\nRules:\n1. Always use get_product_price to retrieve the product's original price.\n2. Always use apply_discount to calculate the discounted price.\n3. Never guess, assume, or invent tool arguments. Only use information explicitly provided by the customer or returned by previous tool calls.\n4. If required information (product name or discount type) is missing, ask the customer before proceeding.\n5. Never perform calculations yourself. All price calculations must be handled by th

In [111]:
run_agent("What is the price of laptop after applying orange discount?")

Question is : What is the price of laptop after applying orange discount? 
The final LLM response:
I can't apply an "orange" discount because it's not a supported discount type. The valid discount types are:

- **gold**: 23% off
- **silver**: 12% off  
- **bronze**: 5% off

Please let me know which discount type you'd like to use (gold, silver, or bronze), and I'll help you find the final price for the laptop.

The message 1 is :
content="You are a shopping assistant. Your job is to help customers find product prices and calculate final prices after discounts.\n\nYou have access to two tools:\n- get_product_price: Retrieves the original price of a product.\n- apply_discount: Applies a discount to a price and returns the final price.\n\nRules:\n\n1-Always use apply_discount to calculate the discounted price.\n2- Always use get_product_price to retrieve the product's original price.\n3. Never guess, assume, or invent tool arguments. Only use information explicitly provided by the custome

In [112]:
from langsmith import traceable
import os
@traceable(name="LangChain agent loop")
def run_agent(question: str):

    tools = [get_product_price , apply_discount]
    tools_by_name = {t.name:t for t in tools}

    llm = init_chat_model(
    model=MODEL,
    model_provider="openai",
    api_key=os.getenv("OpenRouter_API_KEY"),
    base_url="https://openrouter.ai/api/v1",
    temperature=0
)
    model_with_tools = llm.bind_tools(tools)

    print(f"Question is : {question} ")
    print("=" * 40)

    messages = [
        SystemMessage(
           content="""You are a shopping assistant. Your job is to help customers find product prices and calculate final prices after discounts.

You have access to two tools:
- get_product_price: Retrieves the original price of a product.
- apply_discount: Applies a discount to a price and returns the final price.

Rules:

1-Always use apply_discount to calculate the discounted price.
2- Always use get_product_price to retrieve the product's original price.
3. Never guess, assume, or invent tool arguments. Only use information explicitly provided by the customer or returned by previous tool calls.
4. If required information (product name or discount type) is missing, ask the customer before proceeding.
5. Never perform calculations yourself. All price calculations must be handled by the tools.
6. Never invent prices or tool results.
7. If the arguments is not valid ask the user to resend the right argument, give it the valid arguments
After succesfully using both tools, report the product name, original price, discount type, and final price to the customer.
"""
           ),
        HumanMessage(
            content=question
        )
            
    ]
    iteration = 0

    while True:
        iteration += 1

        print(f"\n{'=' * 50}")
        print(f"🤖 MODEL THINKING - Iteration {iteration}")
        print("=" * 50)

        # 1. Send messages to the model
        response = model_with_tools.invoke(messages)

        # 2. Add AI response to conversation history
        messages.append(response)

        print("\n✅ Model response received.")

        # 3. Check whether the model requested any tools
        if not response.tool_calls:
            print("\n" + "=" * 50)
            print("🎯 FINAL ANSWER")
            print("=" * 50)
            print(response.content)
            break

        # 4. Execute requested tools
        for tool_call in response.tool_calls:

            tool_name = tool_call["name"]
            tool_args = tool_call["args"]

            print(f"\n🔧 TOOL REQUESTED: {tool_name}")
            print(f"📥 Arguments: {tool_args}")

            selected_tool = tools_by_name[tool_name]

            print(f"\n⚙️ Executing {tool_name}...")

            result = selected_tool.invoke(tool_args)

            print(f"✅ Tool execution completed.")
            print(f"📤 Tool result: {result}")

            # 5. Send tool result back to conversation
            messages.append(
                ToolMessage(
                    content=str(result),
                    tool_call_id=tool_call["id"]
                )
            )

            print("📨 Tool result added to conversation history.") 

In [113]:
run_agent("What is the price of laptop after applying silver discount?")

Question is : What is the price of laptop after applying silver discount? 

🤖 MODEL THINKING - Iteration 1

✅ Model response received.

🔧 TOOL REQUESTED: get_product_price
📥 Arguments: {'product_name': 'laptop'}

⚙️ Executing get_product_price...
✅ Tool execution completed.
📤 Tool result: 1200.99
📨 Tool result added to conversation history.

🤖 MODEL THINKING - Iteration 2

✅ Model response received.

🔧 TOOL REQUESTED: apply_discount
📥 Arguments: {'discount_type': 'silver', 'price': 1200.99}

⚙️ Executing apply_discount...
✅ Tool execution completed.
📤 Tool result: 1056.87
📨 Tool result added to conversation history.

🤖 MODEL THINKING - Iteration 3

✅ Model response received.

🎯 FINAL ANSWER
Product: Laptop  
Original Price: $1200.99  
Discount Type: Silver  
Final Price: $1056.87
